# ONNX Export and Quantization

This notebook exports a scikit-learn model to ONNX and then quantizes it.

Goals:
- export a model to ONNX
- run it with ONNX Runtime
- compare FP32 and INT8 ONNX files


In [ ]:
%pip install -q onnx onnxruntime skl2onnx

In [ ]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import onnxruntime as ort
from onnxruntime.quantization import QuantType, quantize_dynamic

workdir = Path("/tmp/week11_onnx")
workdir.mkdir(exist_ok=True)

X, y = load_digits(return_X_y=True)
X = X.astype(np.float32)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

clf = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=60, random_state=42)
clf.fit(X_train, y_train)


In [ ]:
initial_type = [("float_input", FloatTensorType([None, X_train.shape[1]]))]
onnx_model = convert_sklearn(clf, initial_types=initial_type)

fp32_path = workdir / "digits_fp32.onnx"
fp32_path.write_bytes(onnx_model.SerializeToString())

int8_path = workdir / "digits_int8.onnx"
quantize_dynamic(str(fp32_path), str(int8_path), weight_type=QuantType.QUInt8)

fp32_session = ort.InferenceSession(str(fp32_path))
int8_session = ort.InferenceSession(str(int8_path))
input_name = fp32_session.get_inputs()[0].name


In [ ]:
def accuracy(session):
    preds = session.run(None, {input_name: X_test})[0]
    return np.mean(preds == y_test)


def benchmark(session, runs=200):
    start = time.perf_counter()
    for _ in range(runs):
        session.run(None, {input_name: X_test})
    return (time.perf_counter() - start) * 1000 / runs


results = pd.DataFrame(
    [
        {
            "model": "onnx_fp32",
            "size_kb": os.path.getsize(fp32_path) / 1024,
            "accuracy": accuracy(fp32_session),
            "latency_ms": benchmark(fp32_session),
        },
        {
            "model": "onnx_int8",
            "size_kb": os.path.getsize(int8_path) / 1024,
            "accuracy": accuracy(int8_session),
            "latency_ms": benchmark(int8_session),
        },
    ]
)
results
